In [ ]:
!pip install pandas torch numpy sklearn statsmodels

In [ ]:
import pandas as pd

df = pd.read_csv("heart_attack_prediction_dataset.csv")
print(df.head())
print(df.describe())

TARGET = 'Heart Attack Risk'

In [ ]:
def retrieve_diastolic_blood_pressure(value):
    return value[value.find('/')+1:]

def retrieve_systolic_blood_pressure(value):
    return value[:value.find('/')]

df['Diastolic Blood Pressure'] = pd.to_numeric(df['Blood Pressure'].map(retrieve_diastolic_blood_pressure))
df['Systolic Blood Pressure'] = pd.to_numeric(df['Blood Pressure'].map(retrieve_systolic_blood_pressure))
print(df['Diastolic Blood Pressure'])
print(df['Systolic Blood Pressure'])

df = df.drop(columns=['Blood Pressure', 'Patient ID'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os
import pandas as pd

def get_numeric_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=[np.number]).columns.tolist()

def get_categorical_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=["object", "category"]).columns.tolist()


def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """Koduje zmienne kategorialne za pomocą Label Encoding."""
    df_encoded = df.copy()
    categorical_cols = get_categorical_columns(df)
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df[col].astype(str))
    
    return df_encoded


def compute_correlation_matrix(df: pd.DataFrame, method: str = "pearson", include_categorical: bool = True) -> pd.DataFrame:
    """Oblicza macierz korelacji, opcjonalnie z uwzględnieniem zmiennych kategorialnych."""
    if include_categorical:
        df_encoded = encode_categorical(df)
        return df_encoded.corr(method=method)
    else:
        numeric_df = df.select_dtypes(include=[np.number])
        return numeric_df.corr(method=method)


def plot_correlation_matrix(
    corr_matrix: pd.DataFrame,
    output_path: str = "outputs/correlation_matrix.png",
    figsize: tuple = (18, 16),
    cmap: str = "coolwarm",
    title: str = "Macierz korelacji wszystkich zmiennych"
):
    plt.figure(figsize=figsize)
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 7}
    )
    
    plt.title(title, fontsize=16, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Macierz korelacji zapisana do: {output_path}")


def get_top_correlations(corr_matrix: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    corr_pairs = corr_matrix.unstack()
    
    corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) < corr_pairs.index.get_level_values(1)]
    
    corr_pairs = corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index)
    
    top_corr = pd.DataFrame({
        "Zmienna 1": [idx[0] for idx in corr_pairs.head(n).index],
        "Zmienna 2": [idx[1] for idx in corr_pairs.head(n).index],
        "Korelacja": corr_pairs.head(n).values
    })
    
    return top_corr

corr_matrix = compute_correlation_matrix(df, method="pearson", include_categorical=True)
plot_correlation_matrix(corr_matrix, output_path="outputs/correlation_matrix.png", title="Macierz korelacji wszystkich zmiennych")
print("Top 10 korelacji:")
print(get_top_correlations(corr_matrix, n=10))

In [ ]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

symbol = ':'

# Q("nazwa_zmiennej") - syntax umożliwiający posługiwanie się pełnymi nazwami kolumn do zdefiniowania modelu liniowego
# C(nazwa_zmiennej) - wskazanie, że dana zmienna jest zmienną kategorialną (jakościową)

categorical_vars = "".join([f'C(Q("{var}")) {symbol} ' for var in get_categorical_columns(df) if var != TARGET])
numeric_vars = "".join([f'{symbol if var != get_numeric_columns(df)[0] else ""} Q("{var}") ' for var in get_numeric_columns(df) if var != TARGET])

definition = f'Q("{TARGET}") ~ ' + categorical_vars + numeric_vars
print(definition)

stats_model = ols(definition, data=df).fit()
anova_result = sm.stats.anova_lm(stats_model, type=2)
print(anova_result)

"""
Najistotniejszą zmienną jest Cholesterol, Hemisphere, Diabetes, Alcohol Consumption i Sleep Hours Per Day, 
jednakże ŻADNA nie jest statystycznie istotna, bo ich PR(>F) jest wyższe od 0.05. 
Wyżej wymienione zmienne to top 5. Skoro nie można zbudować na ich podstawie modelu liniowego, 
to należy albo odnaleźć nieliniowe zależności, albo posłużyć się głęboką siecią neuronową, 
która w kolejnych etapach będzie tworzyć coraz istotniejsze serie danych dla zmiennej zależnej.
"""


In [ ]:
def plot_scatter_plot(df, x, y):
    plt.scatter(df[x], df[y])
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} & {y}")
    os.makedirs(f"outputs/scatterplots/", exist_ok=True)
    plt.savefig(f"outputs/scatterplots/{x} & {y}.png")
    plt.close()

for i in range(len(df.columns)):
    plot_scatter_plot(df, df.columns[i], TARGET)
    # for j in range(i + 1, len(df.columns)):
    #     plot_scatter_plot(df, df.columns[i], df.columns[j])

"""
Niestety, analiza wykresów punktowych nie wykazała żadnych zależności pomiędzy zmienną zależną, a pozostałymi
"""

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

def create_preprocessor(df: pd.DataFrame, target_col: str):
    categorical = df.select_dtypes(include=["object"]).columns.tolist()

    if target_col in categorical:
        categorical.remove(target_col)

    for g in ["passed"]:
        if g in categorical:
            categorical.remove(g)

    numeric = df.select_dtypes(exclude=["object"]).columns.tolist()


    if target_col in numeric:
        numeric.remove(target_col)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
            ("num", StandardScaler(), numeric),
        ]
    )
    return preprocessor

RANDOM_STATE = 42
TEST_SIZE = int(df.__len__() * 0.4)

X = df.drop(columns=[TARGET])
y = df[TARGET]


train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print()

preprocessor = create_preprocessor(df, TARGET)

train_x_t = preprocessor.fit_transform(train_x)
test_x_t = preprocessor.transform(test_x)


le = LabelEncoder()
train_y_t = le.fit_transform(train_y)
test_y_t = le.transform(test_y)



In [ ]:
import torch

x_train_t = torch.tensor(train_x_t.toarray() if hasattr(train_x_t, "toarray") else train_x_t, dtype=torch.float32)
x_test_t = torch.tensor(test_x_t.toarray() if hasattr(test_x_t, "toarray") else test_x_t, dtype=torch.float32)
y_train_t = torch.tensor(train_y_t.reshape(-1, 1), dtype=torch.float32)
y_test_t = torch.tensor(test_y_t.reshape(-1, 1), dtype=torch.float32)

print(x_train_t.shape)



In [ ]:

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = BernoulliNB(alpha=1.0)

nb_model.fit(train_x_t, train_y_t)
y_pred = nb_model.predict(test_x_t)
acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("NB Model - Accuracy: ", acc)
print(report)

lr_model = LogisticRegression(random_state=42)
lr_model.fit(train_x_t, train_y_t)
y_pred = lr_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("LR Model - Accuracy: ", acc)
print(report)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(train_x_t, train_y_t)
y_pred = rf_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)

print("RF Model - Accuracy: ", acc)
print(report)

xgboost_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        random_state=RANDOM_STATE,
        eval_metric="mlogloss"
)
xgboost_model.fit(train_x_t, train_y_t)

y_pred = xgboost_model.predict(test_x_t)

acc = accuracy_score(test_y_t, y_pred)
report = classification_report(test_y_t, y_pred, output_dict=True)


print("XGB Model - Accuracy: ", acc)
print(report)



In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import torch

def plot_roc_auc(y_test, y_score, output_path, model_name="Model"):
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'r--', label='Random Guess')

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Model')
    plt.legend()
    plt.savefig(output_path + " roc auc curve.png")
    plt.close()
    return roc_auc

def normalize_confusion_matrix(cm, norm='true'):
    """
    Normalize a confusion matrix.
    
    Parameters:
    cm (array-like): Confusion matrix to be normalized.
    norm (str): Type of normalization ('true', 'pred', 'all').
    
    Returns:
    ndarray: Normalized confusion matrix.
    """
    if norm == 'true':
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    elif norm == 'pred':
        cm_normalized = cm.astype('float') / cm.sum(axis=0)[np.newaxis, :]
    elif norm == 'all':
        cm_normalized = cm.astype('float') / cm.sum()
    else:
        raise ValueError("Unknown normalization type. Use 'true', 'pred', or 'all'.")
    
    return cm_normalized

def plot_confusion_matrix(y_test, y_pred, output_dir):
    cm = confusion_matrix(y_test, y_pred)
    normalized = normalize_confusion_matrix(cm, norm='true')
    plt.figure(figsize=(5, 4))
    sns.heatmap(normalized, annot=True, fmt=".2f", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_dir), exist_ok=True)
    plt.savefig(output_dir+" confusion matrix.png")
    plt.close()

def plot_training(total_loss, total_acc, epochs, output_path):
    fig, ax = plt.subplots()
    
    ax.plot(epochs, total_loss, color='lightblue', linewidth=3)
    ax.plot(epochs, total_acc, color="red", linewidth=4, marker="o")
    ax.set(xlabel="epochs", ylabel="loss/accuracy")
    
    plt.savefig(output_path+" training.png")
    plt.close()


def train_model(model, x_train_t, y_train_t, epochs=200, lr=0.001, weight_decay=1e-5, logging_step=10, l1_param = 1e-7, output_path = f"outputs/training.png"):
    counts = [counts for _, counts in train_y.value_counts().items()]
    pos_weight = torch.tensor([counts[0] / counts[1]])
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    losses = []
    accuracies = []
    all_epochs = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(x_train_t)
        
        loss = criterion(outputs, y_train_t)
        total_loss = loss + sum(param.abs().sum() for param in model.parameters()) * l1_param
        total_loss.backward()
        optimizer.step()
        if logging_step != -1 and epoch % logging_step == logging_step - 1:
            acc = ((torch.sigmoid(outputs) >= 0.5)== y_train_t.squeeze(1)).float().mean().item()
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Accuracy: {acc:.4f}")

            losses.append(total_loss.item())
            all_epochs.append(epoch+1)
            accuracies.append(acc)
            plot_training(epochs=all_epochs, total_loss=losses, total_acc=accuracies, output_path=output_path)

def evaluate_model(model, x_test_t, y_test_t, le, output_dir, log = True):
    model.eval()
    with torch.no_grad():
        logits = model(x_test_t)
        y_score = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        y_true = y_test_t.squeeze(1).cpu().numpy().astype(int)
        preds = (y_score >= 0.5).astype(int)
        y_pred = le.inverse_transform(preds)
        y_true_labels = le.inverse_transform(y_true)
        acc = np.mean(y_pred == y_true_labels)
        report = classification_report(y_true_labels, y_pred)
        if (log):
            print(f"Accuracy: {acc:.4f}")
            print(report)
            plot_confusion_matrix(y_true_labels, y_pred, output_dir)
            roc_auc = plot_roc_auc(y_true, y_score, output_dir, model.__class__.__name__)
            print(f"ROC AUC: {roc_auc:.4f}")
            
    return acc




In [ ]:
from datetime import datetime

import torch.nn as nn

class DeepNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4, dropout=0.4):
        super(DeepNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_layer, 1)
        )
        

    def forward(self, x):
        x = self.net(x)
        return x
 

In [ ]:
model = DeepNet(x_train_t.shape[1], hidden_layer=128, dropout=0.2)
output_path = f"outputs/{datetime.now().strftime("%H-%M-%S")}"
print(output_path)
print(torch.sigmoid(model(x_test_t[0].unsqueeze(dim=0))), y_test_t[0])
train_model(model, x_train_t, y_train_t, epochs=200, lr=0.001, weight_decay=1e-5, logging_step = 5, l1_param=0, output_path=output_path)

evaluate_model(model, x_test_t, y_test_t, le, output_path)
    

In [ ]:
import torch
import torch.nn as nn

class DeeperNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4, count_of_layers=3):
        super(DeeperNet, self).__init__()
        dense_net = [[nn.Linear(hidden_layer, hidden_layer), nn.ReLU(), nn.Dropout(0.2)] for i in range(count_of_layers - 1)]
        dense_net = [sublayer for layer in dense_net for sublayer in layer]
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            *dense_net,
            nn.Linear(hidden_layer, 1)
        )
        

    def forward(self, x):
        x = self.net(x)
        return x

output_path = f"outputs/{(datetime.now().strftime("%H-%M-%S"))}"
print(x_train_t.shape[1])
model = DeeperNet(x_train_t.shape[1], hidden_layer=256, count_of_layers=10)
train_model(model, x_train_t, y_train_t, epochs=50, lr=0.001, weight_decay=1e-5, logging_step = 5, output_path=output_path, l1_param=0)
evaluate_model(model, x_test_t, y_test_t, le, output_path)

In [ ]:
import optuna

def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 10, 1000)
    dropout = trial.suggest_float("dropout", 0, 1)
    model = DeepNet(x_train_t.shape[1], hidden_layer=hidden_size, dropout=dropout)
    train_model(model, x_train_t, y_train_t, epochs=100, l1_param=0, logging_step=-1)
    return evaluate_model(model, x_test_t, y_test_t, le, output_path, log = False)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)
print(study.best_params)